####  Install and import the necessary computing libraries.

In [1]:
# !pip install librosa
# !pip install pytorch-crf
# !pip install 'praatio<5'
# !pip install matplotlib
# !pip install transformers
# !pip install webrtcvad
# !pip install huggingface_hub[hf_xet]

In [2]:
import os
import glob
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import seaborn as sns
import webrtcvad
import librosa
from librosa.util.exceptions import ParameterError
import soundfile as sf
from datetime import datetime
from torch.utils.data import Dataset, DataLoader, random_split
from torchcrf import CRF
from praatio import tgio
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from transformers import HubertModel, AutoTokenizer, AutoModel, AutoFeatureExtractor, set_seed
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

/Users/moanason/anaconda3/envs/torchpy/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Dataset mapping.

##### Set up the mapping functions

In [3]:
label2id = {"TURN": 0, "<G>": 1, "<B>": 2, "<O>": 3} # label mapping
id2label = {v: k for k, v in label2id.items()}

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

In [6]:
# feature-fitting routines
def prosody_mfcc(x, sr, n_mfcc, hop, n_fft):
    mf = librosa.feature.mfcc(y=x, sr=sr,
                              n_mfcc=n_mfcc,
                              n_fft=n_fft, hop_length=hop).T
    try:
        f0,_,_ = librosa.pyin(
            x,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr, hop_length=hop, frame_length=n_fft
        )
    except Exception:
        f0 = librosa.yin(
            x,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr, hop_length=hop, frame_length=n_fft
        )
    f0   = np.nan_to_num(f0, 0.0)[:mf.shape[0]][:,None]
    rms  = librosa.feature.rms(y=x,
                               hop_length=hop,
                               frame_length=n_fft).T[:mf.shape[0]]
    return np.concatenate([mf, f0, rms], axis=1)


def fit_mfcc_gmms(
    base_dir: str,
    n_components: int = 16,
    frame_duration: float = 0.1,
    sr: int = 44100,
    n_mfcc: int = 13,
    n_fft: int = None
):
    hop_length = int(frame_duration * sr)
    n_fft = n_fft or hop_length

    all_A = []
    all_I = []

    for as_path in glob.glob(os.path.join(base_dir, "rec_*", "*_AS_*.wav")):
        fn   = os.path.basename(as_path)
        part = fn.split("_")[-1].replace(".wav","")
        is_cands = glob.glob(
            os.path.join(os.path.dirname(as_path),
                         f"*_{fn.split('_')[1]}_IS_*_{part}.wav")
        )
        if not is_cands:
            continue
        is_path = is_cands[0]

        # read & align
        A, srA = sf.read(as_path)
        B, srB = sf.read(is_path)
        if A.ndim>1: A = A.mean(axis=1)
        if B.ndim>1: B = B.mean(axis=1)
        if srA!=sr: A = librosa.resample(A, srA, sr)
        if srB!=sr: B = librosa.resample(B, srB, sr)
        L = min(len(A), len(B))
        A, B = A[:L], B[:L]

        # collect prosody_mfcc per channel
        all_A.append(prosody_mfcc(A, sr, n_mfcc, hop_length, n_fft))
        all_I.append(prosody_mfcc(B, sr, n_mfcc, hop_length, n_fft))

    if not all_A or not all_I:
        raise RuntimeError(f"No AS/IS pairs found under {base_dir!r}")

    all_A = np.vstack(all_A)
    all_I = np.vstack(all_I)
    print(f"Fitting GMM_A on {all_A.shape}  —  GMM_I on {all_I.shape}")

    gmm_A = GaussianMixture(n_components=n_components, random_state=42).fit(all_A)
    gmm_I = GaussianMixture(n_components=n_components, random_state=42).fit(all_I)
    return gmm_A, gmm_I

In [7]:
_VAD = webrtcvad.Vad(2)  # or 1–3 aggressiveness

def voice_mask(wav: np.ndarray, sr: int, frame_duration: float, vad_mode:int=3):
    """
    Returns a boolean array of length T = ceil(len(wav)/hop)
    where True = frame is voiced.
    We first resample to 16 kHz (required by webrtcvad), then
    check 30 ms sub-frames and aggregate into your frame_duration bins.
    """
    # 1) resample to 16 kHz
    wav16 = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    hop16 = int(frame_duration * 16000)
    sub_win = 30      # webrtcvad wants 30 ms windows
    sub_n   = int(0.001*sub_win * 16000)
    vad     = webrtcvad.Vad(vad_mode)

    # 2) chunk into sub-frames, run VAD
    is_speech = []
    for i in range(0, len(wav16), sub_n):
        chunk = wav16[i : i+sub_n]
        if len(chunk) < sub_n:
            chunk = np.pad(chunk, (0, sub_n-len(chunk)))
        pcm = (chunk * 32768).astype(np.int16).tobytes()
        is_speech.append(vad.is_speech(pcm, 16000))

    # 3) group sub-frames back into your frame_duration bins
    T = int(np.ceil(len(wav) / int(frame_duration*sr)))
    mask = np.zeros(T, dtype=bool)
    subs_per_bin = int(np.ceil(hop16 / sub_n))
    for t in range(T):
        start, end = t*subs_per_bin, (t+1)*subs_per_bin
        if any(is_speech[start:end]):
            mask[t] = True
    return mask

In [8]:
class GBOAudioDataset(Dataset):
    def __init__(self,
        base_dir: str,
        gmm_A: GaussianMixture,
        gmm_B: GaussianMixture,
        temperature: float = 16.0,
        frame_duration: float = 0.1,
        sr: int = 44100,
        n_mfcc: int = 13,
        chunk_size: int = 500,
        num_chunks: int = 12,
        n_mels: int = 40,
        patch_width: int = 5
    ):
        super().__init__()
        self.base_dir       = base_dir
        self.gmm_A          = gmm_A
        self.gmm_B          = gmm_B
        self.temperature     = temperature
        self.frame_duration = frame_duration
        self.sr             = sr
        self.n_mfcc         = n_mfcc
        self.chunk_size     = chunk_size
        self.num_chunks     = num_chunks
        self.n_mels         = n_mels
        self.patch_width    = patch_width

        # collect (AS.wav, IS.wav, AS.TextGrid, IS.TextGrid)
        self.samples = []
        for dyad in glob.glob(os.path.join(base_dir, "rec_*")):
            ASs = glob.glob(os.path.join(dyad, "*_AS_*.wav"))
            Bs  = glob.glob(os.path.join(dyad, "*_IS_*.wav"))
            for a in ASs:
                part = os.path.basename(a).split("_")[-1].replace(".wav","")
                matches = [b for b in Bs if part in b]
                if not matches: continue
                b = matches[0]
                tgA = a.replace(".wav",".TextGrid")
                tgB = b.replace(".wav",".TextGrid")
                if os.path.exists(tgA) and os.path.exists(tgB):
                    self.samples.append((a,b,tgA,tgB))

        # as_f, is_f, tg_path_A = self.samples[idx]   # this is the path to the .TextGrid
        # tg_path_B = tg_path_A.replace("_AS_", "_IS_")  # or however you get the other TextGrid path


    def __len__(self):
        return len(self.samples)*self.num_chunks

    def __getitem__(self, idx):
        sample_idx = idx//self.num_chunks
        chunk_idx  = idx%self.num_chunks
        a_path, b_path, tgA_path, tgB_path = self.samples[sample_idx]

        # 1) load & align
        A, sA = sf.read(a_path); B, sB = sf.read(b_path)
        if A.ndim>1: A=A.mean(1)
        if B.ndim>1: B=B.mean(1)
        if sA!=self.sr: A=librosa.resample(A,sA,self.sr)
        if sB!=self.sr: B=librosa.resample(B,sB,self.sr)
        L = min(len(A),len(B))
        A,B = A[:L],B[:L]

        # 2) MFCC+prosody → GMM resp
        hop=int(self.frame_duration*self.sr); n_fft=hop
        featsA = prosody_mfcc(A,self.sr,self.n_mfcc,hop,n_fft)
        featsB = prosody_mfcc(B,self.sr,self.n_mfcc,hop,n_fft)

        probA = self.gmm_A.predict_proba(featsA)
        probB = self.gmm_B.predict_proba(featsB)

        # respA  = self.gmm_A.predict_proba(featsA)  # (T_all,n_A)
        # respB  = self.gmm_B.predict_proba(featsB)  # (T_all,n_B)

        T = self.temperature # normalize temperature
        softA = probA ** (1.0 / T)
        softA = softA / softA.sum(axis=1, keepdims=True)
        softB = probB ** (1.0 / T)
        softB = softB / softB.sum(axis=1, keepdims=True)

        # now respA, respB are your new “posterior” features
        respA, respB = softA, softB

        T_all  = respA.shape[0] #identical

        # 3) VAD
        vmA = voice_mask(A,self.sr,self.frame_duration)[:T_all]
        vmB = voice_mask(B,self.sr,self.frame_duration)[:T_all]

        # 4) mel-spectrogram → padded → patches
        mA = librosa.feature.melspectrogram(y=A,sr=self.sr,n_mels=self.n_mels,
                                            n_fft=n_fft,hop_length=hop).T  # (T_all,n_mels)
        mB = librosa.feature.melspectrogram(y=B,sr=self.sr,n_mels=self.n_mels,
                                            n_fft=n_fft,hop_length=hop).T
        pw = self.patch_width
        mA_p = np.pad(mA,   ((0,pw-1),(0,0)), mode='constant')
        mB_p = np.pad(mB,   ((0,pw-1),(0,0)), mode='constant')
        patchesA = np.stack([mA_p[i:i+pw].T for i in range(T_all)],axis=0)
        patchesB = np.stack([mB_p[i:i+pw].T for i in range(T_all)],axis=0)

        # word‐tier tokens
        def _extract_tokens_from_tier(textgrid_path, tier_name, total_frames, frame_duration=0.1):
            """
            Extract tokens from a given tier in a TextGrid file.
            For each frame, return the token if present; otherwise "".
            For the "words" tier, if a token is one of the interference tokens, map it to "".
            """
            interference_tokens = {"<P>", "<RÄUSPERN>", "<SCHMATZEN>", "<LACHEN>", "<UNVERSTÄNDLICH>", "<LÄCHELN>", "<HÄSITATION>", "<GÄHNEN>", "<ATMEN>", "<SEUFZEN>", "<SCHNAUBEN>", "<SCHLUCKEN>", "<HUSTEN>", "NANANA"} 
            try:
                tg = tgio.openTextgrid(textgrid_path)
            except Exception as e:
                print(f"Error reading {textgrid_path}: {e}")
                return [""] * total_frames
            
            if tier_name not in tg.tierDict:
                return [""] * total_frames
            
            tier = tg.tierDict[tier_name]
            tokens = [""] * total_frames
            for (start, end, token) in tier.entryList:
                token = token.strip()
                if token == "":
                    continue
                # for the "words" tier, map interference tokens to ""
                if token.startswith("<") and token.endswith(">"):
                    continue
                if tier_name == "words" and token in interference_tokens:
                    token = ""
                start_frame = int(start / frame_duration)
                end_frame = int(end / frame_duration) + 1
                for i in range(start_frame, min(end_frame, total_frames)):
                    tokens[i] = token
            return tokens

        toksA = _extract_tokens_from_tier(tgA_path, "words", T_all, frame_duration=self.frame_duration)
        toksB = _extract_tokens_from_tier(tgB_path, "words", T_all, frame_duration=self.frame_duration)

        # merged labels from both TextGrids
        def _create_label_sequence(source):
            tg = tgio.openTextgrid(source) if isinstance(source,(str,bytes,os.PathLike)) else source
            labs = ["TURN"] * T_all
            for st,end,lab in tg.tierDict.get("transVals",[]).entryList:
                Lb=lab.strip()
                if Lb == "<L>": continue
                if Lb in {"<G>","<B>","<O>"}:
                    i0,i1 = int(st/self.frame_duration), int(end/self.frame_duration)+1
                    for i in range(i0, min(i1, T_all)): labs[i] = Lb
            return labs
        la = _create_label_sequence(tgA_path)
        lb = _create_label_sequence(tgB_path)
        merged = [a if a!="TURN" else (b if b!="TURN" else "TURN") for a,b in zip(la,lb)]
        labs   = np.array([ label2id.get(x,0) for x in merged ], dtype=np.int64)

        # 7) chunk slice
        start,end=chunk_idx*self.chunk_size,(chunk_idx+1)*self.chunk_size
        respA_c = respA [start:end]
        respB_c = respB [start:end]
        vmA_c   = vmA   [start:end]
        vmB_c   = vmB   [start:end]
        patchesA_c = patchesA[start:end]
        patchesB_c = patchesB[start:end]
        toksA_c = toksA [start:end]
        toksB_c = toksB [start:end]
        labs_c  = labs  [start:end]

        return (
            torch.from_numpy(respA_c).float(),   # (C,n_A)
            torch.from_numpy(respB_c).float(),   # (C,n_B)
            toksA_c, toksB_c,                   # list[str]×2
            torch.from_numpy(vmA_c).bool(),      # (C,)
            torch.from_numpy(vmB_c).bool(),      # (C,)
            torch.from_numpy(patchesA_c).float(),# (C,n_mels,pw)
            torch.from_numpy(patchesB_c).float(),# (C,n_mels,pw)
            torch.from_numpy(labs_c).long()      # (C,)
        )

In [9]:
class GBOAudioDataset_sc(Dataset): # source dataset without chunks
    def __init__(self,
        base_dir: str,
        gmm_A: GaussianMixture,
        gmm_B: GaussianMixture,
        temperature: float = 16.0,
        frame_duration: float = 0.1,
        sr: int = 44100,
        n_mfcc: int = 13,
        # chunk_size: int = 500,
        # num_chunks: int = 12,
        n_mels: int = 40,
        patch_width: int = 5
    ):
        super().__init__()
        self.base_dir       = base_dir
        self.gmm_A          = gmm_A
        self.gmm_B          = gmm_B
        self.temperature     = temperature
        self.frame_duration = frame_duration
        self.sr             = sr
        self.n_mfcc         = n_mfcc
        # self.chunk_size     = chunk_size
        # self.num_chunks     = num_chunks
        self.n_mels         = n_mels
        self.patch_width    = patch_width

        # collect (AS.wav, IS.wav, AS.TextGrid, IS.TextGrid)
        self.samples = []
        for dyad in glob.glob(os.path.join(base_dir, "rec_*")):
            ASs = glob.glob(os.path.join(dyad, "*_AS_*.wav"))
            Bs  = glob.glob(os.path.join(dyad, "*_IS_*.wav"))
            for a in ASs:
                part = os.path.basename(a).split("_")[-1].replace(".wav","")
                matches = [b for b in Bs if part in b]
                if not matches: continue
                b = matches[0]
                tgA = a.replace(".wav",".TextGrid")
                tgB = b.replace(".wav",".TextGrid")
                if os.path.exists(tgA) and os.path.exists(tgB):
                    self.samples.append((a,b,tgA,tgB))

        # as_f, is_f, tg_path_A = self.samples[idx]   # this is the path to the .TextGrid
        # tg_path_B = tg_path_A.replace("_AS_", "_IS_")  # or however you get the other TextGrid path


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # sample_idx = idx//self.num_chunks
        # chunk_idx  = idx%self.num_chunks
        a_path, b_path, tgA_path, tgB_path = self.samples[idx]

        # 1) load & align
        A, sA = sf.read(a_path); B, sB = sf.read(b_path)
        if A.ndim>1: A=A.mean(1)
        if B.ndim>1: B=B.mean(1)
        if sA!=self.sr: A=librosa.resample(A,sA,self.sr)
        if sB!=self.sr: B=librosa.resample(B,sB,self.sr)
        L = min(len(A),len(B))
        A,B = A[:L],B[:L]

        # 2) MFCC+prosody → GMM resp
        hop=int(self.frame_duration*self.sr); n_fft=hop
        featsA = prosody_mfcc(A,self.sr,self.n_mfcc,hop,n_fft)
        featsB = prosody_mfcc(B,self.sr,self.n_mfcc,hop,n_fft)

        probA = self.gmm_A.predict_proba(featsA)
        probB = self.gmm_B.predict_proba(featsB)

        # respA  = self.gmm_A.predict_proba(featsA)  # (T_all,n_A)
        # respB  = self.gmm_B.predict_proba(featsB)  # (T_all,n_B)

        T = self.temperature # normalize temperature
        softA = probA ** (1.0 / T)
        softA = softA / softA.sum(axis=1, keepdims=True)
        softB = probB ** (1.0 / T)
        softB = softB / softB.sum(axis=1, keepdims=True)

        # now respA, respB are your new “posterior” features
        respA, respB = softA, softB

        T_all  = respA.shape[0] #identical

        # 3) VAD
        vmA = voice_mask(A,self.sr,self.frame_duration)[:T_all]
        vmB = voice_mask(B,self.sr,self.frame_duration)[:T_all]

        # 4) mel-spectrogram → padded → patches
        mA = librosa.feature.melspectrogram(y=A,sr=self.sr,n_mels=self.n_mels,
                                            n_fft=n_fft,hop_length=hop).T  # (T_all,n_mels)
        mB = librosa.feature.melspectrogram(y=B,sr=self.sr,n_mels=self.n_mels,
                                            n_fft=n_fft,hop_length=hop).T
        pw = self.patch_width
        mA_p = np.pad(mA,   ((0,pw-1),(0,0)), mode='constant')
        mB_p = np.pad(mB,   ((0,pw-1),(0,0)), mode='constant')
        patchesA = np.stack([mA_p[i:i+pw].T for i in range(T_all)],axis=0)
        patchesB = np.stack([mB_p[i:i+pw].T for i in range(T_all)],axis=0)

        # word‐tier tokens
        def _extract_tokens_from_tier(textgrid_path, tier_name, total_frames, frame_duration=0.1):
            """
            Extract tokens from a given tier in a TextGrid file.
            For each frame, return the token if present; otherwise "".
            For the "words" tier, if a token is one of the interference tokens, map it to "".
            """
            interference_tokens = {"<P>", "<RÄUSPERN>", "<SCHMATZEN>", "<LACHEN>", "<UNVERSTÄNDLICH>", "<LÄCHELN>", "<HÄSITATION>", "<GÄHNEN>", "<ATMEN>", "<SEUFZEN>", "<SCHNAUBEN>", "<SCHLUCKEN>", "<HUSTEN>", "NANANA"} 
            try:
                tg = tgio.openTextgrid(textgrid_path)
            except Exception as e:
                print(f"Error reading {textgrid_path}: {e}")
                return [""] * total_frames
            
            if tier_name not in tg.tierDict:
                return [""] * total_frames
            
            tier = tg.tierDict[tier_name]
            tokens = [""] * total_frames
            for (start, end, token) in tier.entryList:
                token = token.strip()
                if token == "":
                    continue
                # for the "words" tier, map interference tokens to ""
                if token.startswith("<") and token.endswith(">"):
                    continue
                if tier_name == "words" and token in interference_tokens:
                    token = ""
                start_frame = int(start / frame_duration)
                end_frame = int(end / frame_duration) + 1
                for i in range(start_frame, min(end_frame, total_frames)):
                    tokens[i] = token
            return tokens

        toksA = _extract_tokens_from_tier(tgA_path, "words", T_all, frame_duration=self.frame_duration)
        toksB = _extract_tokens_from_tier(tgB_path, "words", T_all, frame_duration=self.frame_duration)

        # merged labels from both TextGrids
        def _create_label_sequence(source):
            tg = tgio.openTextgrid(source) if isinstance(source,(str,bytes,os.PathLike)) else source
            labs = ["TURN"] * T_all
            for st,end,lab in tg.tierDict.get("transVals",[]).entryList:
                Lb=lab.strip()
                if Lb == "<L>": continue
                if Lb in {"<G>","<B>","<O>"}:
                    i0,i1 = int(st/self.frame_duration), int(end/self.frame_duration)+1
                    for i in range(i0, min(i1, T_all)): labs[i] = Lb
            return labs
        la = _create_label_sequence(tgA_path)
        lb = _create_label_sequence(tgB_path)
        merged = [a if a!="TURN" else (b if b!="TURN" else "TURN") for a,b in zip(la,lb)]
        labs   = np.array([ label2id.get(x,0) for x in merged ], dtype=np.int64)

        return (
            torch.from_numpy(respA).float(),   # (C,n_A)
            torch.from_numpy(respB).float(),   # (C,n_B)
            toksA, toksB,                   # list[str]×2
            torch.from_numpy(vmA).bool(),      # (C,)
            torch.from_numpy(vmB).bool(),      # (C,)
            torch.from_numpy(patchesA).float(),# (C,n_mels,pw)
            torch.from_numpy(patchesB).float(),# (C,n_mels,pw)
            torch.from_numpy(labs).long()      # (C,)
        )
        

In [10]:
# def collate_fn(batch):
#     B = len(batch)
#     (rA, rB, tA, tB, vA, vB, mA, mB, lb) = zip(*batch)
#     C_max = max(x.shape[0] for x in rA)
#     nA, nB = rA[0].shape[1], rB[0].shape[1]
#     n_mels,pw = mA[0].shape[1:]

#     # allocate
#     respA_b = torch.zeros(B, C_max, nA)
#     respB_b = torch.zeros(B, C_max, nB)
#     vmA_b   = torch.zeros(B, C_max, dtype=torch.bool)
#     vmB_b   = torch.zeros(B, C_max, dtype=torch.bool)
#     melA_b  = torch.zeros(B, C_max, n_mels, pw)
#     melB_b  = torch.zeros(B, C_max, n_mels, pw)
#     labs_b  = torch.zeros(B, C_max, dtype=torch.long)

#     toksA = []; toksB = []

#     for i,(rAi,rBi,tiA,tiB,vAi,vBi,mAi,mBi,lbi) in enumerate(batch):
#         C = rAi.shape[0]
#         respA_b[i,:C] = rAi
#         respB_b[i,:C] = rBi
#         vmA_b[i,:C]   = vAi
#         vmB_b[i,:C]   = vBi
#         melA_b[i,:C]  = mAi
#         melB_b[i,:C]  = mBi
#         labs_b[i,:C]  = lbi
#         toksA.append(tiA)
#         toksB.append(tiB)

#     return respA_b, respB_b, toksA, toksB, vmA_b, vmB_b, melA_b, melB_b, labs_b

In [11]:
def collate_fn(batch):
    """
    batch is a list of tuples, each tuple = 
      (respA, respB, toksA, toksB, vmA, vmB, patchesA, patchesB, labs)
    where
      respA:      FloatTensor [C_i, n_comp_A]
      respB:      FloatTensor [C_i, n_comp_B]
      toksA:      list of length C_i
      toksB:      list of length C_i
      vmA:        BoolTensor [C_i]
      vmB:        BoolTensor [C_i]
      patchesA:   FloatTensor [C_i, n_mels, pw]
      patchesB:   FloatTensor [C_i, n_mels, pw]
      labs:       LongTensor [C_i]
    We must ensure all of those “C_i” values line up.  If they do not, we clip to the minimum.
    """
    batch_size = len(batch)

    # Step 1: figure out each sample’s “agreed length” Ci = min(len(respA), len(vmA), len(patchesA), len(labs), …)
    lengths = []
    for (respA_i, respB_i, toksA_i, toksB_i, vmA_i, vmB_i, patchesA_i, patchesB_i, labs_i) in batch:
        c1 = respA_i.shape[0]
        c2 = respB_i.shape[0]
        c3 = vmA_i.shape[0]
        c4 = vmB_i.shape[0]
        c5 = patchesA_i.shape[0]
        c6 = patchesB_i.shape[0]
        c7 = labs_i.shape[0]
        # If any of these differ, we forcibly clip them all to the minimum
        Ci = min(c1, c2, c3, c4, c5, c6, c7)
        lengths.append(Ci)

    batch_max = max(lengths)  # We will pad every sequence up to this length

    # Use the first sample to extract feature‐dimension sizes
    respA0, respB0, toksA0, toksB0, vmA0, vmB0, patchesA0, patchesB0, labs0 = batch[0]
    n_comp_A = respA0.shape[1]
    n_comp_B = respB0.shape[1]
    n_mels    = patchesA0.shape[1]
    pw        = patchesA0.shape[2]

    # Allocate zero‐padded tensors of shape [batch_size, batch_max, …]
    respA_b = torch.zeros((batch_size, batch_max, n_comp_A), dtype=torch.float32)
    respB_b = torch.zeros((batch_size, batch_max, n_comp_B), dtype=torch.float32)

    vmA_b = torch.zeros((batch_size, batch_max), dtype=torch.bool)
    vmB_b = torch.zeros((batch_size, batch_max), dtype=torch.bool)

    patchesA_b = torch.zeros((batch_size, batch_max, n_mels, pw), dtype=torch.float32)
    patchesB_b = torch.zeros((batch_size, batch_max, n_mels, pw), dtype=torch.float32)

    labs_b = torch.zeros((batch_size, batch_max), dtype=torch.long)

    toksA_b = []
    toksB_b = []

    # Step 2: copy each sample’s first Ci frames into the respective padded buffer
    for i, (respA_i, respB_i, toksA_i, toksB_i, vmA_i, vmB_i, patchesA_i, patchesB_i, labs_i) in enumerate(batch):
        Ci = lengths[i]

        #  1) respA[0:Ci], respB[0:Ci]
        respA_b[i, :Ci, :] = respA_i[:Ci, :]
        respB_b[i, :Ci, :] = respB_i[:Ci, :]

        #  2) vmA[0:Ci], vmB[0:Ci]
        vmA_b[i, :Ci] = vmA_i[:Ci]
        vmB_b[i, :Ci] = vmB_i[:Ci]

        #  3) patchesA[0:Ci], patchesB[0:Ci]
        patchesA_b[i, :Ci, :, :] = patchesA_i[:Ci, :, :]
        patchesB_b[i, :Ci, :, :] = patchesB_i[:Ci, :, :]

        #  4) labs[0:Ci]
        labs_b[i, :Ci] = labs_i[:Ci]

        #  5) toksA_i and toksB_i are length Ci; we keep them as Python lists of length Ci
        toksA_b.append(toksA_i[:Ci])
        toksB_b.append(toksB_i[:Ci])

    return (
        respA_b,    # [B, batch_max, n_comp_A]
        respB_b,    # [B, batch_max, n_comp_B]
        toksA_b,    # Python list of length B; each entry is a list of length Ci
        toksB_b,    # Python list of length B
        vmA_b,      # [B, batch_max] (bool)
        vmB_b,      # [B, batch_max] (bool)
        patchesA_b, # [B, batch_max, n_mels, pw]
        patchesB_b, # [B, batch_max, n_mels, pw]
        labs_b      # [B, batch_max] (long)
    )


##### Fit GMM mfcc.

In [12]:
base_dir = "/Users/moanason/Downloads/GBO_audio_train"

gmm_AS, gmm_IS = fit_mfcc_gmms(
    base_dir=base_dir,
    n_components=16,
    frame_duration=0.1,
    sr=44100,
    n_mfcc=13
)

Fitting GMM_A on (18003, 15)  —  GMM_I on (18003, 15)


/Users/moanason/anaconda3/envs/torchpy/lib/python3.9/site-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/Users/moanason/anaconda3/envs/torchpy/lib/python3.9/site-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


##### Concatenate the dataset.

In [13]:
dataset = GBOAudioDataset_sc(base_dir,
                          gmm_A=gmm_AS,
                          gmm_B=gmm_IS,
                          frame_duration=0.1,
                          sr=44100,
                          n_mfcc=13,
                        #   chunk_size=500,
                          # num_chunks=12
)

print("Total samples:", len(dataset))

Total samples: 3


Check the data structure.

In [14]:
# for i in range(1):
#     respA, respB, toksA, toksB, vmA, vmB, melA, melB, labs = dataset[i]
#     print(f"#{i:02d}: "
#           f"feats_A={respA.shape}, toks_A={len(toksA)}, vmA={vmA.shape}, melA={melA.shape}; "
#           f"feats_I={respB.shape}, toks_I={len(toksB)}, vmI={vmB.shape}, melB={melB.shape}; "
#           f"labs={labs.shape}")

In [15]:
# for j in range(150):
#     print(f"\nFrame #{j}")
#     print("  respA =", respA[j].tolist())
#     print("  tokenA =", toksA[j])
#     print("  vmA =", bool(vmA[j].item()))
#     print("  melA =", melA[j].tolist())
#     print("  respI =", respB[j].tolist())
#     print("  tokenI =", toksB[j])
#     print("  vmI =", bool(vmB[j].item()))
#     print("  melI =", melB[j].tolist())
#     print("  label =", labs[j].item())

#### BiLSTM Architecture with CRF and cross attention with BERT.

In [16]:
class NonSpeechCNN(nn.Module):
    def __init__(self, n_mels=40, patch_width=5):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,16,(3,3),padding=1), nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(16,32,(3,3),padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc   = nn.Linear(32,3)  # silence / noise / other

    def forward(self,x):
        # x: (B·C,1,n_mels,pw)
        h=self.conv(x).flatten(1)     # (B·C,32)
        return self.fc(h)            # (B·C,3)

In [17]:
# class BiLSTMCRFWithBertCrossAttention(nn.Module):
#     def __init__(
#         self,
#         n_comp_A: int,
#         n_comp_I: int,
#         hidden_dim: int,
#         n_labels: int,
#         bert_model: str = "dbmdz/bert-base-german-uncased",
#         max_len: int = 6001,
#         num_layers: int = 4,
#         dropout: float = 0.1,
#         num_heads: int = 4,
#         n_mels:int=40, pw:int=5
#     ):
#         super().__init__()
#         # Store n_mels and patch_width as instance attributes
#         self.n_mels = n_mels
#         self.patch_width = pw

#         # — audio stacks: 4‐layer bidir LSTMs —
#         # Use a single multi-layer LSTM module for each stream
#         self.lstmA = nn.LSTM(
#             input_size=n_comp_A,
#             hidden_size=hidden_dim,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True,
#             dropout=dropout if num_layers > 1 else 0 # Dropout only applied between layers if num_layers > 1
#         )
#         self.lstmI = nn.LSTM(
#             input_size=n_comp_I,
#             hidden_size=hidden_dim,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True,
#             dropout=dropout if num_layers > 1 else 0 # Dropout only applied between layers if num_layers > 1
#         )
#         self.drop = nn.Dropout(dropout) # Apply dropout after the LSTM stack

#         # — residual projections —
#         # Layers to project the initial input dimensions to the LSTM output dimensions (2*hidden_dim)
#         # for the residual connection after the LSTM stack.
#         self.input_residual_projA = nn.Linear(n_comp_A, 2*hidden_dim)
#         self.input_residual_projI = nn.Linear(n_comp_I, 2*hidden_dim)


#         # — BERT text encoder & tokenizer —
#         self.tokenizer    = AutoTokenizer.from_pretrained(bert_model)
#         self.text_encoder = AutoModel.from_pretrained(bert_model)
#         d_text = self.text_encoder.config.hidden_size  # e.g. 768

#         # — project audio → BERT‐dim + positional encoding —
#         # Project the output of the LSTM (2*hidden_dim) to the BERT dimension (d_text)
#         self.audio_projA = nn.Linear(2*hidden_dim, d_text)
#         self.audio_projI = nn.Linear(2*hidden_dim, d_text)

#         # Positional encoding: Using the learned parameter approach from original code
#         # Ensure max_len is sufficient for the sequence length (T, which is ds.chunk_size)
#         self.pos_enc     = nn.Parameter(torch.zeros(1, max_len, d_text))
#         nn.init.trunc_normal_(self.pos_enc, std=0.02)

#         # — per‐channel cross‐attention decoder layers —
#         dec_layerA = nn.TransformerDecoderLayer(
#             d_model=d_text, nhead=num_heads, dropout=dropout, batch_first=True
#         )
#         dec_layerI = nn.TransformerDecoderLayer(
#             d_model=d_text, nhead=num_heads, dropout=dropout, batch_first=True
#         )
#         self.crossA = nn.TransformerDecoder(dec_layerA, num_layers=1)
#         self.crossI = nn.TransformerDecoder(dec_layerI, num_layers=1)

#         # NonSpeechCNN and projection
#         self.ns_cnn  = NonSpeechCNN(n_mels,pw)
#         # NonSpeechCNN output is 3 dimensions, project it to d_text
#         self.ns_proj = nn.Linear(3, d_text)

#         # — final classifier + CRF —
#         self.classifier = nn.Linear(d_text, n_labels)
#         self.crf        = CRF(n_labels, batch_first=True)

#     def forward(
#         self,
#         respA, respB, toksA, toksB, vmA, vmB, melA, melB
#     ):
#         """
#         respA: (B, T, n_comp_A)
#         respB: (B, T, n_comp_I)
#         toksA: list of B lists of length T (strings)
#         toksB: list of B lists of length T (strings)
#         vmA:   (B, T) bool mask
#         vmB:   (B, T) bool mask
#         melA:  (B, T, n_mels, pw)
#         melB:  (B, T, n_mels, pw)
#         """
#         B, T, _ = respA.size()
#         device = respA.device

#         # — audio stacks: Pass inputs directly to multi-layer LSTMs —
#         # The nn.LSTM module handles multi-layers internally.
#         # The output of the LSTM will be (B, T, 2*hidden_dim)
#         outA, _ = self.lstmA(respA)
#         outI, _ = self.lstmI(respB)

#         # — Apply dropout after LSTM stack —
#         # Apply dropout to the final output of the LSTM stack
#         outA = self.drop(outA)
#         outI = self.drop(outI)

#         # — Apply residual connections from initial input to final LSTM output —
#         # Project the initial inputs to match the LSTM output dimensions (2*hidden_dim)
#         resA = self.input_residual_projA(respA)
#         resI = self.input_residual_projI(respB)
#         # Add the projected initial input to the LSTM output
#         outA = outA + resA
#         outI = outI + resI


#         # — project into BERT‐dim + positional encoding —
#         # Project the LSTM outputs (now including residual) to the BERT dimension
#         xA = self.audio_projA(outA)
#         xI = self.audio_projI(outI)

#         # Add positional encoding.
#         # Ensure the sequence length T does not exceed the maximum length defined for positional encoding.
#         if T > self.pos_enc.size(1):
#              raise ValueError(f"Sequence length T ({T}) exceeds max_len ({self.pos_enc.size(1)}) for positional encoding.")

#         # Add positional encoding to the projected audio features
#         xA = xA + self.pos_enc[:, :T, :]
#         xI = xI + self.pos_enc[:, :T, :]

#         # Remove the extra and incorrect positional encoding application and typo
#         # xA = self.pos_enc(xA); xB = self.pos_enc(xB) # REMOVE THIS LINE

#         # Non-speech CNN
#         # Reshape melA and melB from (B, T, n_mels, pw) to (B*T, 1, n_mels, pw) for the CNN
#         MA = melA.unsqueeze(1).view(B * T, 1, self.n_mels, self.patch_width)
#         MB = melB.unsqueeze(1).view(B * T, 1, self.n_mels, self.patch_width)

#         nsA = self.ns_cnn(MA)  # Output shape (B*T, 3)
#         nsB = self.ns_cnn(MB)  # Output shape (B*T, 3)

#         # Project CNN output (3) to d_text and reshape back to (B, T, d_text)
#         nsA = self.ns_proj(nsA).view(B, T, -1)
#         nsB = self.ns_proj(nsB).view(B, T, -1)

#         # Add the non-speech features to the audio features after positional encoding and LSTM+Residual
#         xA = xA + nsA
#         xI = xI + nsB

#         # — tokenize & encode text for both speakers —
#         # This part tokenizes the word sequences and feeds them through the BERT model.
#         # The output txtA/txtI shape is (B, seq_len_bert, d_text)
#         # maskA/maskI shape is (B, seq_len_bert) boolean

#         encA = self.tokenizer(
#             toksA,
#             is_split_into_words=True,
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#             max_length=512 # Max sequence length for BERT
#         ).to(device)
#         # Get the last hidden states from BERT
#         txtA = self.text_encoder(**encA).last_hidden_state
#         maskA = encA["attention_mask"].bool() # Attention mask for BERT input

#         encI = self.tokenizer(
#             toksB,
#             is_split_into_words=True,
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#             max_length=512 # Max sequence length for BERT
#         ).to(device)
#         # Get the last hidden states from BERT
#         txtI = self.text_encoder(**encI).last_hidden_state
#         maskI = encI["attention_mask"].bool() # Attention mask for BERT input

#         # — cross‐attention per channel —
#         # tgt=xA (B, T, d_text) - audio features
#         # memory=txtA (B, seq_len_bert, d_text) - text features
#         # memory_key_padding_mask=~maskA (B, seq_len_bert) - mask for text
#         # The TransformerDecoder performs cross-attention where audio queries text.
#         # Output yA, yI are (B, T, d_text)
#         yA = self.crossA(tgt=xA, memory=txtA, memory_key_padding_mask=~maskA)
#         yI = self.crossI(tgt=xI, memory=txtI, memory_key_padding_mask=~maskI)

#         # — merge streams & classify —
#         # Combine the outputs from cross-attention (simple averaging)
#         y = 0.5 * (yA + yI)
#         # Pass the combined features through the final classifier
#         logits = self.classifier(y)  # Output shape (B, T, n_labels)
#         return logits

#     def compute_loss(self, logits, tags, mask=None):
#         """
#         Computes the negative log-likelihood loss using the CRF layer.
#         Args:
#             logits (torch.Tensor): Output logits from the classifier (B, T, n_labels).
#             tags (torch.Tensor): Ground truth labels (B, T).
#             mask (torch.Tensor, optional): Mask tensor (B, T) indicating valid frames.
#                                           Defaults to a mask of all True.
#         Returns:
#             torch.Tensor: The negative log-likelihood loss.
#         """
#         if mask is None:
#             # Create a mask if none is provided (all frames are considered valid)
#             mask = torch.ones_like(tags, dtype=torch.bool)
#         # The CRF layer takes logits, ground truth tags, and an optional mask.
#         # Reduction="mean" calculates the mean loss over the batch.
#         return -self.crf(logits, tags, mask=mask, reduction="mean")

#     def decode(self, logits, mask=None):
#         """
#         Decodes the best sequence of labels using the Viterbi algorithm in the CRF layer.
#         Args:
#             logits (torch.Tensor): Output logits from the classifier (B, T, n_labels).
#             mask (torch.Tensor, optional): Mask tensor (B, T) indicating valid frames.
#                                           Defaults to a mask of all True.
#         Returns:
#             list: A list of predicted label sequences for each sample in the batch.
#                   Each sequence is a list of integers.
#         """
#         if mask is None:
#             # Create a mask if none is provided (all frames are considered valid)
#             mask = torch.ones(logits.size(0), logits.size(1), dtype=torch.bool, device=logits.device)
#         # The CRF decode method returns the best tag sequence.
#         return self.crf.decode(logits, mask=mask)

In [18]:
class BiLSTMWithBertCrossAttention(nn.Module):
    def __init__(
        self,
        n_comp_A: int,
        n_comp_I: int,
        hidden_dim: int,
        n_labels: int,
        bert_model: str = "dbmdz/bert-base-german-uncased",
        max_len: int = 6001,
        num_layers: int = 4,
        dropout: float = 0.1,
        num_heads: int = 4,
        n_mels: int = 40,
        pw: int = 5
    ):
        super().__init__()
        # Store n_mels and patch_width as instance attributes
        self.n_mels = n_mels
        self.patch_width = pw

        # — audio stacks: 4‐layer bidir LSTMs —
        self.lstmA = nn.LSTM(
            input_size=n_comp_A,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.lstmI = nn.LSTM(
            input_size=n_comp_I,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.drop = nn.Dropout(dropout)

        # — residual projections —
        self.input_residual_projA = nn.Linear(n_comp_A, 2 * hidden_dim)
        self.input_residual_projI = nn.Linear(n_comp_I, 2 * hidden_dim)

        # — BERT text encoder & tokenizer —
        self.tokenizer    = AutoTokenizer.from_pretrained(bert_model)
        self.text_encoder = AutoModel.from_pretrained(bert_model)
        self.d_text       = self.text_encoder.config.hidden_size 

        # — project audio → BERT‐dim + positional encoding —
        self.audio_projA = nn.Linear(2 * hidden_dim, self.d_text)
        self.audio_projI = nn.Linear(2 * hidden_dim, self.d_text)

        # Learned positional encoding (size = max_len × d_text)
        self.pos_enc = nn.Parameter(torch.zeros(1, max_len, self.d_text))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)

        # — per‐channel cross‐attention decoder layers —
        dec_layerA = nn.TransformerDecoderLayer(
            d_model=self.d_text, nhead=num_heads, dropout=dropout, batch_first=True
        )
        dec_layerI = nn.TransformerDecoderLayer(
            d_model=self.d_text, nhead=num_heads, dropout=dropout, batch_first=True
        )
        self.crossA = nn.TransformerDecoder(dec_layerA, num_layers=1)
        self.crossI = nn.TransformerDecoder(dec_layerI, num_layers=1)

        # NonSpeechCNN and projection (unchanged from before)
        self.ns_cnn  = NonSpeechCNN(n_mels, pw)
        self.ns_proj = nn.Linear(3, self.d_text)

        # 6) fusion‐transformer over the two streams
        fuse_layer = nn.TransformerEncoderLayer(d_model=self.d_text,
                                                nhead=num_heads,
                                                dropout=dropout,
                                                batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(fuse_layer, num_layers=1)

        # — final linear classifier (CTC does not need CRF) —
        self.classifier = nn.Linear(self.d_text, n_labels)

        # — replace CRF with CTC —
        # We assume “0” is reserved as the CTC “blank” index
        self.ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

    def forward(
        self,
        respA, respB, toksA, toksB, vmA, vmB, melA, melB
    ):
        """
        Inputs:
          respA: (B, T, n_comp_A)
          respB: (B, T, n_comp_I)
          toksA, toksB: lists of B lists, each length T
          vmA, vmB:   (B, T) boolean masks (not used for forward ‐ but could be used upstream)
          melA, melB: (B, T, n_mels, pw)
        """
        B, T, _ = respA.size()
        device = respA.device

        # — 1) BiLSTM on each audio stream —
        outA, _ = self.lstmA(respA)    # (B, T, 2*hidden_dim)
        outI, _ = self.lstmI(respB)    # (B, T, 2*hidden_dim)

        outA = self.drop(outA)
        outI = self.drop(outI)

        # — 2) residual from raw GMM‐resp → project → add —
        resA = self.input_residual_projA(respA)  # (B, T, 2*hidden_dim)
        resI = self.input_residual_projI(respB)
        outA = outA + resA
        outI = outI + resI

        # — 3) project into BERT‐dim & add positional encoding —
        xA = self.audio_projA(outA)   # (B, T, d_text)
        xI = self.audio_projI(outI)   # (B, T, d_text)

        if T > self.pos_enc.size(1):
            raise ValueError(f"Sequence length T ({T}) exceeds max_len ({self.pos_enc.size(1)})")

        xA = xA + self.pos_enc[:, :T, :]
        xI = xI + self.pos_enc[:, :T, :]

        # — 4) NonSpeechCNN on mel patches → project → add back —
        #    reshape to (B*T, 1, n_mels, pw) for CNN
        MA = melA.unsqueeze(1).view(B * T, 1, self.n_mels, self.patch_width)
        MB = melB.unsqueeze(1).view(B * T, 1, self.n_mels, self.patch_width)

        nsA = self.ns_cnn(MA)       # (B*T, 3)
        nsB = self.ns_cnn(MB)       # (B*T, 3)

        nsA = self.ns_proj(nsA).view(B, T, -1)  # (B, T, d_text)
        nsB = self.ns_proj(nsB).view(B, T, -1)

        xA = xA + nsA
        xI = xI + nsB

        # — 5) tokenize & BERT‐encode text for both speakers —
        encA = self.tokenizer(
            toksA,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)
        txtA = self.text_encoder(**encA).last_hidden_state  # (B, L₁, d_text)
        maskA = encA["attention_mask"].bool()               # (B, L₁)

        encI = self.tokenizer(
            toksB,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)
        txtB = self.text_encoder(**encI).last_hidden_state  # (B, L₂, d_text)
        maskB = encI["attention_mask"].bool()               # (B, L₂)

        # --- cross-attention audio→text
        yA = self.crossA(tgt=xA,
                         memory=txtA,
                         memory_key_padding_mask=~maskA)
        yI = self.crossI(tgt=xI,
                         memory=txtB,
                         memory_key_padding_mask=~maskB)

        # --- merge streams + classify → CRF
        y_stack = torch.stack([yA, yI], dim=2)      # (B, T, 2, d_text)
        y_stack = y_stack.view(B, T*2, self.d_text) # (B, 2T, d_text)
        y_fused = self.fusion_transformer(y_stack)  # (B, 2T, d_text)
        y = y_fused.view(B, T, 2, self.d_text).mean(2)   # (B, T, d_text)

        logits = self.classifier(y)      # (B, w, n_labels)
        return logits

    # def compute_loss(self, logits, tags, mask=None):
    #     """
    #     Replace the CRF‐loss with a CTC‐loss.

    #     Args:
    #       logits:  (B, T, n_labels)  — raw scores (no softmax)
    #       tags:    (B, T)            — each frame’s true label (0..n_labels-1)
    #       mask:    (B, T) boolean    — if you only want to train on certain frames

    #     Returns:
    #       a single scalar loss = CTC( log_probs, targets, input_lengths, target_lengths )
    #     """
    #     B, T, C = logits.size()
    #     device = logits.device

    #     # If no mask given, assume all True
    #     if mask is None:
    #         mask = torch.ones((B, T), dtype=torch.bool, device=device)

    #     # 1) Compute log-probabilities for CTC
    #     #    CTC in PyTorch expects shape (T, B, C) in log-space
    #     log_probs = logits.log_softmax(dim=-1)           # (B, T, C)
    #     log_probs = log_probs.transpose(0, 1)             # (T, B, C)

    #     # 2) Build “input_lengths” = a length‐vector of size B.
    #     #    Since we are using all T time‐steps, we set each = T.
    #     input_lengths = torch.full(
    #         (B,), T, dtype=torch.long, device=device
    #     )  # shape (B,)

    #     # 3) Build “target” as a 1D vector. 
    #     #    For CTC, you need to flatten out your “tags” and pass a single 1D tensor,
    #     #    along with the “target_lengths” vector. Here we assume every position in tags is “valid,”
    #     #    so each target length = T as well.
    #     #
    #     #    If you only want to supervise on a subset of frames (using mask), you would first
    #     #    remove the masked‐out positions. For simplicity below, we only do full‐length.
    #     targets = tags.view(-1).to(dtype=torch.long)  # (B*T,)
    #     target_lengths = torch.full(
    #         (B,), T, dtype=torch.long, device=device
    #     )  # (B,)

    #     # 4) Finally call CTC
    #     loss = self.ctc_loss(log_probs,       # (T, B, C) log‐probs
    #                          targets,         # (B*T,)  flattened labels
    #                          input_lengths,  # (B,)
    #                          target_lengths  # (B,)
    #                          )
    #     return loss

    def compute_loss(self, logits, tags, mask=None):
        """
        Compute CTC loss properly by extracting only the non‐zero (non‐blank)
        label IDs from each example’s tags[t=0..T-1].
        Args:
            logits: Tensor of shape (B, T, C) – raw scores for each class including blank=0
            tags:   Tensor of shape (B, T) – integer label IDs in [0..C-1],
                    where 0 represents “blank” (no label) and >0 are real labels.
            mask:   Optional (B,T) boolean – True for frames that should be counted,
                    False to ignore entirely (e.g. padding).  If None → assume all True.
        Returns:
            A single‐scalar CTC loss (on whichever device `logits` lives).
        """
        device = logits.device
        B, T, C = logits.size()

        if mask is None:
            mask = torch.ones((B, T), dtype=torch.bool, device=device)

        # Step 1) log_probs: (T, B, C)
        log_probs = logits.log_softmax(dim=-1).transpose(0, 1)  # (T, B, C)

        # Step 2) For each batch element b, collect its target labels (non‐zero up to mask):
        targets_list = []          # will hold all non‐zero labels for entire batch
        target_lengths = []        # U_b for each batch element

        # We also need input_lengths: how many time‐steps are valid per example
        input_lengths = mask.sum(dim=1).long()  # (B,) – number of “True” frames

        for b in range(B):
            # Only consider the frames where mask[b, t] == True, up to T
            valid_frames = mask[b]           # Boolean mask shape (T,)
            all_tags = tags[b]               # shape (T,)
            # Extract only those tags where (valid_frames == True) AND (tag != 0)
            true_labels = all_tags[valid_frames].long()  # (num_valid,)
            # Keep only non‐zero labels (0 = blank)
            true_labels = true_labels[true_labels != 0]  # remove zero entries

            # If no non-zero labels exist, then the “target” is length=0
            target_lengths.append(true_labels.numel())
            if true_labels.numel() > 0:
                targets_list.append(true_labels)

        # If every batch element has zero actual labels (e.g. all‐blank),
        # CTC expects `targets_cat` to be an empty tensor.
        if len(targets_list) == 0:
            # Build an empty int64 tensor on CPU (CTCLoss expects CPU targets sometimes)
            targets_cat = torch.empty((0,), dtype=torch.long)
        else:
            # Concatenate all target sequences in a single 1‐D
            targets_cat = torch.cat(targets_list).cpu()  # move concat’d labels to CPU

        # Convert to normal Python lists
        input_lengths_cpu  = input_lengths.cpu()  # (B,)
        target_lengths_cpu = torch.tensor(target_lengths, dtype=torch.long)

        # Step 3) Compute CTC loss **on CPU** (since MPS may not implement ctc_loss).
        log_probs_cpu = log_probs.cpu().contiguous()  # (T, B, C) on CPU

        loss_cpu = self.ctc_loss(
            log_probs_cpu,
            targets_cat,
            input_lengths_cpu,
            target_lengths_cpu
        )

        # Step 4) Move scalar back to original device
        return loss_cpu.to(device)

    def decode(self, logits, mask=None):
        """
        A simple “best‐path” decode: take argmax over classes at each time step,
        then collapse repeats and drop blanks. This is *not* the Viterbi algorithm
        of a CRF—just a greedy CTC decoding.
        """
        if mask is None:
            mask = torch.ones(logits.size(0), logits.size(1), dtype=torch.bool, device=logits.device)

        # 1) pick most likely class at each time step
        argmax_ids = logits.argmax(dim=-1)  # (B, T)
        results = []
        for b in range(logits.size(0)):
            seq = []
            prev = None
            for t in range(logits.size(1)):
                if not mask[b, t]:
                    break
                ix = argmax_ids[b, t].item()
                # 0 is blank → skip
                if ix != 0 and ix != prev:
                    seq.append(ix)
                prev = ix
            results.append(seq)
        return results


In [19]:
class PositionalEncodingBatchFirst(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=6001):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

#### Training loop.

##### Training with VAD mask

In [ ]:
def train_model(ds,
                gmm_as,
                gmm_is,
                device,
                num_epochs: int = 50,
                batch_size: int = 2,
                patience: int = 5,
                checkpoint_dir: str = "/Users/moanason/Downloads/GBO/models/checkpoints_gmm"):
    os.makedirs(checkpoint_dir, exist_ok=True)

    n = len(ds); ntr=int(0.8*n)
    tr,val = random_split(ds,[ntr,n-ntr])
    loader_tr = DataLoader(tr, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    loader_va = DataLoader(val, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model = BiLSTMWithBertCrossAttention(
        n_comp_A=gmm_as.n_components,
        n_comp_I=gmm_is.n_components,
        hidden_dim=64,
        n_labels=4,
        bert_model="dbmdz/bert-base-german-uncased",
        max_len=ds.chunk_size,
        n_mels=40, pw=5
    ).to(device)

    opt = optim.Adam(model.parameters(), lr=5e-3)
    # Initialize best_val_loss and epochs_no_imp
    best_val_loss = float('inf')
    epochs_no_imp = 0


    for e in range(1,num_epochs+1):
        # — train —
        model.train(); tot_tr=0
        for respA,respB,tA,tB,vmA,vmB,mA,mB,labs in loader_tr:
            respA,respB=respA.to(device),respB.to(device)
            vmA,  vmB  =vmA.to(device),  vmB.to(device)
            mA, mB     =mA.to(device), mB.to(device)
            labs       =labs.to(device)

            mask = (vmA|vmB); mask[:,0]=True
            opt.zero_grad()
            # Correct the order of arguments passed to the model
            logits = model(respA, respB, tA, tB, vmA, vmB, mA, mB)
            loss   = model.compute_loss(logits,labs,mask)
            loss.backward(); opt.step()
            tot_tr+=loss.item()

        # — valid —
        model.eval(); tot_va=0
        with torch.no_grad():
            for respA,respB,tA,tB,vmA,vmB,mA,mB,labs in loader_va:
                respA,respB=respA.to(device),respB.to(device)
                vmA,  vmB  =vmA.to(device),  vmB.to(device)
                mA, mB     =mA.to(device), mB.to(device)
                labs       =labs.to(device)

                mask=(vmA|vmB); mask[:,0]=True
                 # Correct the order of arguments passed to the model
                logits=model(respA, respB, tA, tB, vmA, vmB, mA, mB)
                tot_va+=model.compute_loss(logits,labs,mask).item()

        avg_tr=tot_tr/len(loader_tr); avg_val=tot_va/len(loader_va)
        print(f"[{e:02d}] train={avg_tr:.4f} val={avg_val:.4f}")

        # — early‐stop & checkpoint best
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            epochs_no_imp = 0
            best_path = os.path.join(checkpoint_dir, "model_crf_gbo_gmm11.pt")
            torch.save(model.state_dict(), best_path)
            print(f"  → New best (val_loss={avg_val:.4f}), saved to {best_path}")
        else:
            epochs_no_imp += 1
            if epochs_no_imp >= patience:
                print(f"  → Early stopping after {patience} epochs w/o improvement.")
                break

    # final save
    final_path = os.path.join(checkpoint_dir, "model_crf_gbo_gmm2.pt")
    torch.save(model.state_dict(), final_path)
    print(f"Training complete. Final model saved to {final_path}")

    return model

##### Training without VAD mask

In [20]:
def train_model(
    ds,
    gmm_as,
    gmm_is,
    device,
    num_epochs: int = 50,
    batch_size: int = 1,
    patience: int = 5,
    checkpoint_dir: str = "/Users/moanason/Downloads/GBO/models/checkpoints_gmm",
):
    os.makedirs(checkpoint_dir, exist_ok=True)

    # 80/20 split
    n = len(ds)
    n_train = int(0.8 * n)
    train_ds, val_ds = random_split(ds, [n_train, n - n_train])
    loader_tr = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    loader_va = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # instantiate model
    model = BiLSTMWithBertCrossAttention(
        n_comp_A   = gmm_as.n_components,
        n_comp_I   = gmm_is.n_components,
        hidden_dim = 64,
        n_labels   = 4,
        bert_model = "dbmdz/bert-base-german-uncased",
        max_len    = 6001
        # max_len    = ds.chunk_size
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=5e-3)
    best_val_loss = float('inf')
    epochs_no_imp = 0

    for epoch in range(1, num_epochs + 1):
        # — train —
        model.train()
        total_train = 0.0
        for respA, respB, toksA, toksB, vmA, vmB, mA, mB, labs in loader_tr:
            # move to device
            respA, respB = respA.to(device), respB.to(device)
            mA,   mB     = mA.to(device),  mB.to(device)
            labs         = labs.to(device)

            # forward
            logits = model(respA, respB, toksA, toksB, None, None, mA, mB)

            # compute loss on ALL frames
            loss = model.compute_loss(logits, labs)  # mask=None → all‐true internally
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_train += loss.item()

        avg_tr = total_train

        # — validate —
        model.eval()
        total_val = 0.0
        with torch.no_grad():
            for respA, respB, toksA, toksB, vmA, vmB, mA, mB, labs in loader_va:
                respA, respB = respA.to(device), respB.to(device)
                mA,   mB     = mA.to(device),  mB.to(device)
                labs         = labs.to(device)

                logits = model(respA, respB, toksA, toksB, None, None, mA, mB)
                total_val += model.compute_loss(logits, labs).item()

        avg_val = total_val
        print(f"[Epoch {epoch:02d}] train_loss={avg_tr:.4f}  val_loss={avg_val:.4f}")

        # — early‐stop & checkpoint best —
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            epochs_no_imp = 0
            best_path = os.path.join(checkpoint_dir, "model_gbo_nc_ctc_k3.pt") # no vad, trial 13
            torch.save(model.state_dict(), best_path)
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"  → New best | val_loss={avg_val:.4f}, saved to {best_path} @ {ts}")
        else:
            epochs_no_imp += 1
            if epochs_no_imp >= patience:
                print(f"  → Early stopping after {patience} epochs w/o improvement.")
                break

    # final save
    final_path = os.path.join(checkpoint_dir, "model_gbo_nc_ctc_k2.pt")
    torch.save(model.state_dict(), final_path)
    print(f"Training complete. Final model saved to {final_path}")

    return model


##### Training call

In [ ]:
if __name__ == "__main__":
    device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
    print("Device:", device)

    model = train_model(
        ds=dataset,
        gmm_as=gmm_AS,
        gmm_is=gmm_IS,
        device=device,
        num_epochs=50,
        batch_size=1,
        patience=5,
        checkpoint_dir="/Users/moanason/Downloads/GBO/models/checkpoints_gmm")

In [ ]:
print(os.getcwd())
print(os.listdir('.'))

#### Evaluation with confusion matrix.

##### With VAD

In [ ]:
def evaluate_model(test_ds,
                   model_path: str,
                   device: torch.device,
                   batch_size: int = 2):

    # 1) build our test loader
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    # 2) pull model‐config from dataset
    n_comp_A = test_ds.gmm_A.n_components
    n_comp_I = test_ds.gmm_B.n_components
    n_mels   = test_ds.n_mels   # Get n_mels from the dataset
    pw       = test_ds.patch_width # Get patch_width from the dataset


    # 3) re-instantiate and load weights
    model = BiLSTMWithBertCrossAttention(
        n_comp_A   = n_comp_A,
        n_comp_I   = n_comp_I,
        hidden_dim = 64,
        n_labels   = len(label2id),
        bert_model = "dbmdz/bert-base-german-uncased",
        max_len    = test_ds.chunk_size,  # or your full T if not chunked
        n_mels     = n_mels, # Pass n_mels to the model
        pw         = pw      # Pass patch_width to the model
    ).to(device)

    ckpt = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt)
    model.eval()

    all_gold = []
    all_pred = []

    # 4) iterate
    with torch.no_grad():
        # Unpack all 9 items returned by collate_fn
        for respA, respB, toksA, toksB, vmA, vmB, melA, melB, labs in test_loader:
            # move to device
            respA, respB = respA.to(device), respB.to(device)
            vmA,   vmB   = vmA.to(device),   vmB.to(device)
            melA,  melB  = melA.to(device),  melB.to(device) # Move mel spectrograms to device
            labs         = labs.to(device)

            # build evaluation mask: any voiced frame in either channel
            # This is the mask used to filter gold labels and final predictions
            original_eval_mask = vmA | vmB   # (B, T)

            # Mask for CRF decoding: First timestep must be True
            # This mask is only used by model.decode
            crf_decode_mask = original_eval_mask.clone() # Clone to avoid modifying original_eval_mask
            crf_decode_mask[:, 0] = True

            # forward + viterbi
            logits = model(
                respA, respB, toksA, toksB, None, vmB, melA, melB
            )                        # (B, T, n_labels)

            # Decode using the mask that forces the first frame to be True.
            # The output `batch_preds` is a list of lists. Each inner list
            # contains predictions ONLY for the frames where `crf_decode_mask` was True.
            batch_preds = model.decode(logits, mask=crf_decode_mask)

            B, T = labs.shape
            for b in range(B):
                # The predicted sequence from model.decode is filtered by crf_decode_mask.
                # We need to re-align these predictions to the original sequence length (T)
                # and then filter based on the original_eval_mask.

                # Create a full-length prediction sequence (padded with a dummy value, e.g., -1)
                # and fill in the actual predictions at the indices where crf_decode_mask was True.
                full_pred_seq = [-1] * T # Use -1 as a placeholder for masked frames

                # Get the indices where crf_decode_mask was True
                masked_indices = torch.where(crf_decode_mask[b])[0].tolist()

                # Check if the number of predictions matches the number of masked indices
                if len(batch_preds[b]) != len(masked_indices):
                     # This should not happen with the current logic, but good for debugging
                     print(f"Warning: Decode length mismatch for batch item {b}. Decoded length: {len(batch_preds[b])}, Masked indices count: {len(masked_indices)}")


                # Place the predictions from batch_preds[b] into the full_pred_seq at the correct original indices
                for i_masked, original_idx in enumerate(masked_indices):
                    if original_idx < T: # Ensure index is within bounds of the original sequence length
                       full_pred_seq[original_idx] = batch_preds[b][i_masked]
                    # else: # Handle potential indices beyond original T if crf_decode_mask was True but original_eval_mask wasn't
                        # This case is unlikely given the masks are derived from T, but adding robustness
                        # print(f"Warning: Original index {original_idx} out of bounds ({T}) for batch item {b}.")


                # Now, filter both gold and the *re-aligned* predicted sequences
                # using the original_eval_mask (mask indicating frames where either speaker was voiced).
                # This aligns evaluation to only the intended voiced segments.
                gold_seq_filtered = labs[b, :T][original_eval_mask[b, :T]].cpu().tolist()
                pred_seq_filtered = [full_pred_seq[i] for i in range(T) if original_eval_mask[b, i]]

                # Ensure lengths match (sanity check)
                if len(gold_seq_filtered) != len(pred_seq_filtered):
                     print(f"Warning: Length mismatch after filtering for batch item {b}. Gold length: {len(gold_seq_filtered)}, Pred length: {len(pred_seq_filtered)}")
                     # If this warning appears, there might still be a subtle mismatch
                     # between how masks affect CRF decoding and subsequent filtering.
                     # However, forcing the first frame mask to True is necessary for CRF.

                all_gold.extend(gold_seq_filtered)
                all_pred.extend(pred_seq_filtered)


    # 5) compute and display metrics
    acc     = accuracy_score(all_gold, all_pred)
    macro_f = f1_score(all_gold, all_pred, average="macro", zero_division=0) # Add zero_division for safety
    # Ensure labels for confusion matrix cover all possible labels in gold and predicted
    # Use the sorted keys of label2id to ensure consistent label order
    labels_for_cm = sorted(label2id.values())
    cm      = confusion_matrix(all_gold, all_pred, labels=labels_for_cm)
    names   = [id2label[i] for i in labels_for_cm] # Use labels_for_cm for names
    report  = classification_report(all_gold, all_pred, target_names=names, zero_division=0)

    print(f"Token-level Accuracy: {acc:.4f}")
    print(f"Macro F1 Score:       {macro_f:.4f}\n")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=names, yticklabels=names
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Test Confusion Matrix")
    plt.tight_layout()
    plt.show()

    return acc, macro_f, cm, report

##### Without VAD.

In [ ]:

def evaluate_model(
    test_ds: GBOAudioDataset,
    model_path: str,
    device: torch.device,
    batch_size: int = 2,
):
    """
    Evaluate a CRF‐based model on *all* frames (no VAD masks).
    Assumes Dataset.__getitem__ returns exactly:
        respA, respB, toksA, toksB, vmA, vmB, melA, melB, labs
    We ignore vmA/vmB and replace them with "all‐true". 
    """

    # 1) DataLoader over the test dataset
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    # 2) Re‐create & load the trained model
    #    We extract n_comp_A, n_comp_I, etc. from test_ds.gmm_A / gmm_B so that
    #    the architecture matches exactly what you used during training.
    n_comp_A = test_ds.gmm_A.n_components
    n_comp_I = test_ds.gmm_B.n_components
    hidden_dim = 64
    n_labels   = len(label2id)
    bert_model = "dbmdz/bert-base-german-uncased"
    max_len    = test_ds.chunk_size      # same chunk_size as training
    num_layers = 4
    dropout    = 0.1
    num_heads  = 4
    n_mels     = test_ds.n_mels          # same as training
    pw         = test_ds.patch_width     # same as training

    model = BiLSTMWithBertCrossAttention(
        n_comp_A   = n_comp_A,
        n_comp_I   = n_comp_I,
        hidden_dim = hidden_dim,
        n_labels   = n_labels,
        bert_model = bert_model,
        max_len    = max_len,
        num_layers = num_layers,
        dropout    = dropout,
        num_heads  = num_heads,
        n_mels     = n_mels,
        pw         = pw
    ).to(device)

    # Load saved state dict
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    # 3) Iterate over test set and accumulate all gold vs. predicted labels
    all_gold = []
    all_pred = []

    with torch.no_grad():
        for (respA, respB, toksA, toksB, vmA, vmB, melA, melB, labs) in test_loader:
            # Move everything to the chosen device
            respA = respA.to(device)   # (B, T, n_comp_A)
            respB = respB.to(device)   # (B, T, n_comp_I)
            melA  = melA.to(device)    # (B, T, n_mels, pw)
            melB  = melB.to(device)    # (B, T, n_mels, pw)
            labs  = labs.to(device)    # (B, T)
            B, T = labs.size()

            # Build a mask of shape (B, T) that's all True, since we want to score
            # every frame (no VAD).  dtype=torch.bool
            all_true = torch.ones((B, T), dtype=torch.bool, device=device)

            # Forward pass: supply all_true in place of vmA/vmB
            logits = model(
                respA, respB,
                toksA, toksB,
                all_true, all_true,   # vmA, vmB  → all‐true
                melA, melB
            )
            # logits has shape (B, T, n_labels)

            # Viterbi‐decode over all frames (mask=all_true)
            # returns a Python list of length B, each entry is a list[int] of length T
            batch_preds = model.decode(logits, mask=all_true)

            # Append to flat lists
            for b in range(B):
                gold_seq = labs[b].detach().cpu().tolist()  # list of length T
                pred_seq = batch_preds[b]                  # list of length T

                # Extend the global lists
                all_gold.extend(gold_seq)
                all_pred.extend(pred_seq)


    # 4) Compute metrics
    labels_sorted = sorted(label2id.values())                     # e.g. [0,1,2,3]
    names_sorted  = [id2label[i] for i in labels_sorted]           # e.g. ["None","<G>","<B>","<O>"]

    acc     = accuracy_score(all_gold, all_pred)
    macro_f = f1_score(all_gold, all_pred, average="macro", zero_division=0)
    cm      = confusion_matrix(all_gold, all_pred, labels=labels_sorted)
    report  = classification_report(
        all_gold,
        all_pred,
        labels=labels_sorted,
        target_names=names_sorted,
        zero_division=0
    )

    print(f"\n── Evaluation on ALL frames ───────────────────────────────────")
    print(f"Token‐level Accuracy: {acc:.4f}")
    print(f"Macro F1 Score:      {macro_f:.4f}\n")

    print("Classification Report:")
    print(report)

    print("Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=names_sorted,
        yticklabels=names_sorted
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix (all frames)")
    plt.tight_layout()
    plt.show()

    return acc, macro_f, cm, report

##### Neue

In [50]:
def extract_nonzero_labels(
    labels_tensor: torch.Tensor,
    mask: torch.Tensor = None
) -> list:
    """
    Given `labels_tensor` of shape (C,) (integers in [0..n_labels-1], where 0=blank),
    optionally apply a Boolean mask of shape (C,) to keep only positions mask==True,
    then collect all nonzero entries in order, returning them as a Python list of ints.
    """
    if mask is not None:
        valid = labels_tensor[mask]
    else:
        valid = labels_tensor
    arr = valid.detach().cpu().numpy().tolist()
    return [int(x) for x in arr if int(x) != 0]

def evaluate_model(
    test_ds: GBOAudioDataset,
    model_path: str,
    device: torch.device,
    batch_size: int = 2,
):
    """
    Evaluate a CTC‐based BiLSTM+BERT+CRF model on the entire test set, with no VAD
    (i.e. we force every frame to be “valid”).  We:
      1) instantiate a DataLoader over test_ds using its collate_fn,
      2) re‐create the same model architecture and load its saved state_dict,
      3) for each batch, do a forward pass, CTC‐decode (greedy), and compare the
         predicted label sequence vs. the ground‐truth nonzero labels,
      4) compute both sequence‐level exact match rate and average edit distance,
      5) print a confusion matrix on a per‐label “flattened” basis (optional).
    """

    # 1) Build DataLoader (no shuffling!)
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    # 2) Re‐instantiate & load your model exactly as in training:
    n_comp_A = test_ds.gmm_A.n_components
    n_comp_I = test_ds.gmm_B.n_components
    n_labels = len(label2id)        # 0..(N-1), where 0 is blank
    max_len  = 6001
    n_mels   = test_ds.n_mels
    pw       = test_ds.patch_width

    model = BiLSTMWithBertCrossAttention(
        n_comp_A   = n_comp_A,
        n_comp_I   = n_comp_I,
        hidden_dim = 64,
        n_labels   = n_labels,
        bert_model = "dbmdz/bert-base-german-uncased",
        max_len    = max_len,
        num_layers = 4,
        dropout    = 0.1,
        num_heads  = 4,
        n_mels     = n_mels,
        pw         = pw
    ).to(device)

    ckpt = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt)
    model.eval()

    # 3) Iterate over the test set, decode, collect sequences
    total_windows  = 0
    exact_matches  = 0
    edit_dist_list = []

    for (respA, respB, toksA, toksB, vmA, vmB, melA, melB, labs) in test_loader:
        # Move tensors to device
        respA = respA.to(device)  # shape (B, C, n_comp_A)
        respB = respB.to(device)  # shape (B, C, n_comp_I)
        melA  = melA.to(device)   # shape (B, C, n_mels, pw)
        melB  = melB.to(device)   # shape (B, C, n_mels, pw)
        labs  = labs.to(device)   # shape (B, C)

        B, C = labs.shape

        # We do NOT use the provided vmA/vmB masks at all.  Instead, force “all frames valid”:
        all_true = torch.ones((B, C), dtype=torch.bool, device=device)

        # Forward pass: note our `forward` signature is
        #    forward(respA, respB, toksA, toksB, vmA, vmB, melA, melB)
        with torch.no_grad():
            logits = model(
                respA, respB,
                toksA, toksB,
                all_true, all_true,   # replace any real VAD masks with “all True”
                melA, melB
            )
            # logits: shape (B, C, n_labels)

            # CTC‐decode each window (greedy + remove repeated/blanks)
            batch_preds = model.decode(logits, mask=all_true)
            # batch_preds[b] is a Python list of ints (all >0, since blanks=0 are removed)

            # Now extract the ground‐truth nonzero labels per example
            for b in range(B):
                gt_seq   = extract_nonzero_labels(labs[b], mask=all_true[b])
                pred_seq = batch_preds[b]

                # Exact match?
                if pred_seq == gt_seq:
                    exact_matches += 1

                # Record Levenshtein edit distance
                d = editdistance.eval(pred_seq, gt_seq)
                edit_dist_list.append(d)

                total_windows += 1

    # 4) Compute and print summary metrics
    seq_accuracy = exact_matches / total_windows * 100.0
    mean_edit    = float(np.mean(edit_dist_list))

    print("──────────────────────────────────────────────")
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Evaluation Summary:")
    print(f"  • # test windows           = {total_windows}")
    print(f"  • Sequence exact‐match (%) = {seq_accuracy:.2f}%")
    print(f"  • Avg. edit distance       = {mean_edit:.2f}")
    print("──────────────────────────────────────────────")

    return {
        "num_windows": total_windows,
        "seq_exact_match_pct": seq_accuracy,
        "avg_edit_distance": mean_edit
    }

##### Evaluation call

In [ ]:
class GBOAudioDataset_sc(Dataset): # source dataset without chunks
    def __init__(self,
        base_dir: str,
        gmm_A: GaussianMixture,
        gmm_B: GaussianMixture,
        temperature: float = 16.0,
        frame_duration: float = 0.1,
        sr: int = 44100,
        n_mfcc: int = 13,
        # chunk_size: int = 500,
        # num_chunks: int = 12,
        n_mels: int = 40,
        patch_width: int = 5
    ):
        super().__init__()
        self.base_dir       = base_dir
        self.gmm_A          = gmm_A
        self.gmm_B          = gmm_B
        self.temperature     = temperature
        self.frame_duration = frame_duration
        self.sr             = sr
        self.n_mfcc         = n_mfcc
        # self.chunk_size     = chunk_size
        # self.num_chunks     = num_chunks
        self.n_mels         = n_mels
        self.patch_width    = patch_width

        # collect (AS.wav, IS.wav, AS.TextGrid, IS.TextGrid)
        self.samples = []
        for dyad in glob.glob(os.path.join(base_dir, "rec_*")):
            ASs = glob.glob(os.path.join(dyad, "*_AS_*.wav"))
            Bs  = glob.glob(os.path.join(dyad, "*_IS_*.wav"))
            for a in ASs:
                part = os.path.basename(a).split("_")[-1].replace(".wav","")
                matches = [b for b in Bs if part in b]
                if not matches: continue
                b = matches[0]
                tgA = a.replace(".wav",".TextGrid")
                tgB = b.replace(".wav",".TextGrid")
                if os.path.exists(tgA) and os.path.exists(tgB):
                    self.samples.append((a,b,tgA,tgB))

        # as_f, is_f, tg_path_A = self.samples[idx]   # this is the path to the .TextGrid
        # tg_path_B = tg_path_A.replace("_AS_", "_IS_")  # or however you get the other TextGrid path


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # sample_idx = idx//self.num_chunks
        # chunk_idx  = idx%self.num_chunks
        a_path, b_path, tgA_path, tgB_path = self.samples[idx]

        # 1) load & align
        A, sA = sf.read(a_path); B, sB = sf.read(b_path)
        if A.ndim>1: A=A.mean(1)
        if B.ndim>1: B=B.mean(1)
        if sA!=self.sr: A=librosa.resample(A,sA,self.sr)
        if sB!=self.sr: B=librosa.resample(B,sB,self.sr)
        L = min(len(A),len(B))
        A,B = A[:L],B[:L]

        # 2) MFCC+prosody → GMM resp
        hop=int(self.frame_duration*self.sr); n_fft=hop
        featsA = prosody_mfcc(A,self.sr,self.n_mfcc,hop,n_fft)
        featsB = prosody_mfcc(B,self.sr,self.n_mfcc,hop,n_fft)

        probA = self.gmm_A.predict_proba(featsA)
        probB = self.gmm_B.predict_proba(featsB)

        # respA  = self.gmm_A.predict_proba(featsA)  # (T_all,n_A)
        # respB  = self.gmm_B.predict_proba(featsB)  # (T_all,n_B)

        T = self.temperature # normalize temperature
        softA = probA ** (1.0 / T)
        softA = softA / softA.sum(axis=1, keepdims=True)
        softB = probB ** (1.0 / T)
        softB = softB / softB.sum(axis=1, keepdims=True)

        # now respA, respB are your new “posterior” features
        respA, respB = softA, softB

        T_all  = respA.shape[0] #identical

        # 3) VAD
        vmA = voice_mask(A,self.sr,self.frame_duration)[:T_all]
        vmB = voice_mask(B,self.sr,self.frame_duration)[:T_all]

        # 4) mel-spectrogram → padded → patches
        mA = librosa.feature.melspectrogram(y=A,sr=self.sr,n_mels=self.n_mels,
                                            n_fft=n_fft,hop_length=hop).T  # (T_all,n_mels)
        mB = librosa.feature.melspectrogram(y=B,sr=self.sr,n_mels=self.n_mels,
                                            n_fft=n_fft,hop_length=hop).T
        pw = self.patch_width
        mA_p = np.pad(mA,   ((0,pw-1),(0,0)), mode='constant')
        mB_p = np.pad(mB,   ((0,pw-1),(0,0)), mode='constant')
        patchesA = np.stack([mA_p[i:i+pw].T for i in range(T_all)],axis=0)
        patchesB = np.stack([mB_p[i:i+pw].T for i in range(T_all)],axis=0)

        # word‐tier tokens
        def _extract_tokens_from_tier(textgrid_path, tier_name, total_frames, frame_duration=0.1):
            """
            Extract tokens from a given tier in a TextGrid file.
            For each frame, return the token if present; otherwise "".
            For the "words" tier, if a token is one of the interference tokens, map it to "".
            """
            interference_tokens = {"<P>", "<RÄUSPERN>", "<SCHMATZEN>", "<LACHEN>", "<UNVERSTÄNDLICH>", "<LÄCHELN>", "<HÄSITATION>", "<GÄHNEN>", "<ATMEN>", "<SEUFZEN>", "<SCHNAUBEN>", "<SCHLUCKEN>", "<HUSTEN>", "NANANA"} 
            try:
                tg = tgio.openTextgrid(textgrid_path)
            except Exception as e:
                print(f"Error reading {textgrid_path}: {e}")
                return [""] * total_frames
            
            if tier_name not in tg.tierDict:
                return [""] * total_frames
            
            tier = tg.tierDict[tier_name]
            tokens = [""] * total_frames
            for (start, end, token) in tier.entryList:
                token = token.strip()
                if token == "":
                    continue
                # for the "words" tier, map interference tokens to ""
                if token.startswith("<") and token.endswith(">"):
                    continue
                if tier_name == "words" and token in interference_tokens:
                    token = ""
                start_frame = int(start / frame_duration)
                end_frame = int(end / frame_duration) + 1
                for i in range(start_frame, min(end_frame, total_frames)):
                    tokens[i] = token
            return tokens

        toksA = _extract_tokens_from_tier(tgA_path, "words", T_all, frame_duration=self.frame_duration)
        toksB = _extract_tokens_from_tier(tgB_path, "words", T_all, frame_duration=self.frame_duration)

        # merged labels from both TextGrids
        def _create_label_sequence(source):
            tg = tgio.openTextgrid(source) if isinstance(source,(str,bytes,os.PathLike)) else source
            labs = ["TURN"] * T_all
            for st,end,lab in tg.tierDict.get("transVals",[]).entryList:
                Lb=lab.strip()
                if Lb == "<L>": continue
                if Lb in {"<G>","<B>","<O>"}:
                    i0,i1 = int(st/self.frame_duration), int(end/self.frame_duration)+1
                    for i in range(i0, min(i1, T_all)): labs[i] = Lb
            return labs
        la = _create_label_sequence(tgA_path)
        lb = _create_label_sequence(tgB_path)
        merged = [a if a!="TURN" else (b if b!="TURN" else "TURN") for a,b in zip(la,lb)]
        labs   = np.array([ label2id.get(x,0) for x in merged ], dtype=np.int64)

        return (
            torch.from_numpy(respA).float(),   # (C,n_A)
            torch.from_numpy(respB).float(),   # (C,n_B)
            toksA, toksB,                   # list[str]×2
            torch.from_numpy(vmA).bool(),      # (C,)
            torch.from_numpy(vmB).bool(),      # (C,)
            torch.from_numpy(patchesA).float(),# (C,n_mels,pw)
            torch.from_numpy(patchesB).float(),# (C,n_mels,pw)
            torch.from_numpy(labs).long()      # (C,)
        )
        

In [37]:
if __name__ == "__main__":
    test_base_dir = "/Users/moanason/Downloads/GBO_audio_eval"
    gmm_A_ts, gmm_I_ts = fit_mfcc_gmms(test_base_dir)


Fitting GMM_A on (6001, 15)  —  GMM_I on (6001, 15)


/Users/moanason/anaconda3/envs/torchpy/lib/python3.9/site-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


In [52]:
if __name__ == "__main__":

    test_ds = GBOAudioDataset_eval(test_base_dir, gmm_A_ts, gmm_I_ts)

    print("Total samples:", len(test_ds))

Total samples: 1


In [ ]:
for i in range(1):
    respA, respB, toksA, toksB, vmA, vmB, melA, melB, labs = test_ds[i]
    print(f"#{i:02d}: "
          f"feats_A={respA.shape}, toks_A={len(toksA)}, vmA={vmA.shape}, melA={melA.shape}; "
          f"feats_I={respB.shape}, toks_I={len(toksB)}, vmI={vmB.shape}, melB={melB.shape}; "
          f"labs={labs.shape}")

#00: feats_A=torch.Size([6001, 16]), toks_A=6001, vmA=torch.Size([6000]), melA=torch.Size([6001, 40, 5]); feats_I=torch.Size([6001, 16]), toks_I=6001, vmI=torch.Size([6000]), melB=torch.Size([6001, 40, 5]); labs=torch.Size([6001])


In [43]:
for j in range(250):
    print(f"\nFrame #{j}")
    print("  respA =", respA[j].tolist())
    print("  tokenA =", toksA[j])
    print("  vmA =", bool(vmA[j].item()))
    print("  melA =", melA[j].tolist())
    print("  respI =", respB[j].tolist())
    print("  tokenI =", toksB[j])
    print("  vmI =", bool(vmB[j].item()))
    print("  melI =", melB[j].tolist())
    print("  label =", labs[j].item())


Frame #0
  respA = [0.0002828666183631867, 0.0, 1.107930510101078e-17, 0.0346701517701149, 0.0, 0.0, 0.0, 0.11356831341981888, 5.202408503990341e-10, 0.002237746026366949, 0.7049562335014343, 0.0013670165790244937, 4.3584384457062697e-07, 0.0007630831096321344, 0.13471518456935883, 0.007438985630869865]
  tokenA = Plastikbecher
  vmA = True
  melA = [[0.13025207817554474, 0.8989478349685669, 0.272691935300827, 0.1682889759540558, 0.33530500531196594], [0.003753082361072302, 25.80962371826172, 14.768961906433105, 0.0282712671905756, 6.232329368591309], [0.0010448769899085164, 6.510382175445557, 1.2123801708221436, 0.0021631624549627304, 1.616194486618042], [0.0005200343439355493, 1.5773823261260986, 0.1598004847764969, 0.000772609084378928, 0.965879499912262], [0.00042273238068446517, 1.6648731231689453, 0.029303845018148422, 0.000311695272102952, 0.4427700340747833], [0.00028152577579021454, 1.8407182693481445, 0.120407834649086, 0.0001196349912788719, 0.0662441998720169], [0.00025224

In [44]:
import editdistance

In [ ]:
model_path = "/Users/moanason/Downloads/GBO/models/checkpoints_gmm/model_gbo_ctc_k1.pt"
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("Device:", device)

results = evaluate_model(
    test_ds    = test_ds,
    model_path = model_path,
    device     = device,
    batch_size = 2
)
print("Returned results:", results)